# Figure 1 — HOMER learns a calibrated probabilistic mouse–human coupling

**What §1 claims.** HOMER solves a fused Gromov–Wasserstein optimal-transport problem and returns a
coupling π (1,864 mouse parcels × 2,094 human parcels). Every mouse parcel gets a *probability
distribution* over the human brain, not a single asserted partner. Three properties make it usable:

1. **It is sharp.** The top human partner carries a median probability of 1.0 and exceeds 0.5 for 92 %
   of mouse parcels.
2. **It is homology-respecting.** Aggregated to the 21 Garin homology classes, mass concentrates on the
   diagonal — mean self-mass 0.26, about five times the 0.048 expected under a uniform mapping.
3. **It is confidence-graded.** Each parcel carries an evidence tier, and the tier predicts *the
   resolution at which the prediction can be trusted*.

Plus one negative control that matters: **trust cannot be read from the solver.** At the production
regularisation the coupling is sharply peaked everywhere, so the solver's own confidence is uncorrelated
with accuracy. The evidence grades are external, not internal.

> **The rule this notebook follows.** Every statistic is read from, or checked against, the JSON that
> `manuscript/results_section.md` cites for §1: `coupling_summary.json`, `fig1_coupling_matrix.json`,
> `evidence_tiers_v2.json`. An earlier draft of §1 carried tier percentages that summed to **101 %** and
> contradicted its own Extended Data caption. Numbers do not get typed here.

In [ ]:
import sys, json, warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy.stats import pearsonr

warnings.filterwarnings('ignore')
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
sys.path.insert(0, str(ROOT.parent / 'manuscript' / 'figures'))

from homer.data import load_cached, load_pi
# coarse_region maps each parcel to one of the 21 Garin homology classes. It is the ONE piece of
# shared machinery this notebook imports: the class assignment must be identical to the one the
# figure script uses, or the self-mass number changes meaning.
from _common import coarse_region, row_normalise, GARIN_NAMES

LOGS = ROOT / 'outputs' / 'logs'

pi = load_pi()                            # pi_fc_plus_SC_with_all_packs.npy — the recommended coupling
M, _ = load_cached('mouse', cache_dir=str(ROOT / 'outputs/anndata'))
H, _ = load_cached('human', cache_dir=str(ROOT / 'outputs/anndata'))

cs = json.loads((LOGS / 'coupling_summary.json').read_text())
fc = json.loads((LOGS / 'fig1_coupling_matrix.json').read_text())
ev = json.loads((LOGS / 'evidence_tiers_v2.json').read_text())


def check(name, computed, expected, tol):
    ok = abs(computed - expected) <= tol
    print(f"  {'OK      ' if ok else 'MISMATCH'}  {name}: notebook {computed:.4f}  vs  log {expected:.4f}")
    assert ok, f'{name} diverged from the canonical log'


print(f'pi: {pi.shape[0]} mouse x {pi.shape[1]} human parcels')
check('pi shape (mouse)', pi.shape[0], cs['pi_shape'][0], 0)
check('pi shape (human)', pi.shape[1], cs['pi_shape'][1], 0)

## 1. The coupling is sharp (Fig. 1d)

Row-normalise π so each mouse parcel becomes a probability distribution over the 2,094 human parcels,
then ask how much mass its single best human partner carries.

In [ ]:
P = row_normalise(pi)                      # rows sum to 1
top_p = P.max(axis=1)
med = float(np.median(top_p))
frac_sharp = float((top_p > 0.5).mean())

print(f'top-target probability: median {med:.4f}, mean {top_p.mean():.3f}')
print(f'{frac_sharp * 100:.1f} % of mouse parcels place > 0.5 of their mass on a single human parcel')
print()
check('median top-target probability', med, cs['top_target_probability']['median'], 1e-6)
check('fraction > 0.5', frac_sharp, cs['top_target_probability']['fraction_above_0.5'], 1e-6)

In [ ]:
# ---------------- Fig 1d — the argmax diagonal ----------------
amax = P.argmax(axis=1)
fig, ax = plt.subplots(figsize=(7.0, 6.2))
sc = ax.scatter(amax, np.arange(pi.shape[0]), s=6, c=top_p, cmap='viridis',
                vmin=0, vmax=1, linewidths=0)
ax.set_xlim(0, pi.shape[1])
ax.set_ylim(pi.shape[0], 0)
ax.set_xlabel('human parcel (of 2,094)')
ax.set_ylabel('mouse parcel (of 1,864)')
cb = fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.02)
cb.set_label('top-target probability')
ax.set_title('Each mouse parcel at its most probable human partner\n'
             f'median top-target probability {med:.2f};  > 0.5 for {frac_sharp * 100:.0f} % of parcels',
             fontweight='bold', loc='left', fontsize=10.5)
plt.show()

print('The argmax falls on a clean diagonal. That diagonal is NOT trivially guaranteed: parcels are')
print('indexed in anatomical order in both species, so an orderly diagonal means routing preserves')
print('topography. Section 3 below tests that directly, rather than eyeballing it.')

## 2. Mass concentrates on the homologous diagonal (Fig. 1e)

Aggregate π to the 21 Garin homology classes and row-normalise. The **diagonal** is the fraction of each
mouse class's mass that lands on its own human class.

> The self-mass number is 0.26. It is computed **here**, with `coarse_region()`, exactly as the figure
> script computes it. A nearest-anchor recomputation gives 0.275 — a *different quantity*. During the
> audit 0.26 was briefly "corrected" to 0.275 on that basis. The script that builds a panel owns that
> panel's numbers; do not recompute them with a different definition.

In [ ]:
mc = coarse_region(M.var)                  # class 1..21 per mouse parcel
hc = coarse_region(H.var)                  # class 1..21 per human parcel

K = 21
Cmat = np.zeros((K, K))
for i in range(1, K + 1):
    rows = np.where(mc == i)[0]
    if not len(rows):
        continue
    for j in range(1, K + 1):
        Cmat[i - 1, j - 1] = pi[np.ix_(rows, np.where(hc == j)[0])].sum()
Crow = Cmat / np.maximum(Cmat.sum(axis=1, keepdims=True), 1e-12)

self_mass = float(np.mean(np.diag(Crow)))
uniform = 1.0 / K
print(f'mean self-mass on the homologous diagonal : {self_mass:.4f}')
print(f'expected under a uniform mapping          : {uniform:.4f}')
print(f'fold enrichment                           : {self_mass / uniform:.2f}x')
print()
check('mean self-mass', self_mass, fc['mean_self_mass_diagonal'], 1e-6)
check('fold over uniform', self_mass / uniform, fc['fold_over_uniform'], 1e-4)

In [ ]:
# ---------------- Fig 1e — the 21 x 21 class coupling ----------------
NAMES = [GARIN_NAMES[i] for i in range(1, K + 1)]   # GARIN_NAMES is a dict keyed 1..21
fig, ax = plt.subplots(figsize=(8.2, 7.0))
im = ax.imshow(np.clip(Crow, 1e-4, None), cmap='magma', norm=LogNorm(vmin=1e-3, vmax=1.0))
ax.set_xticks(range(K)); ax.set_xticklabels(NAMES, rotation=90, fontsize=7.5)
ax.set_yticks(range(K)); ax.set_yticklabels(NAMES, fontsize=7.5)
ax.set_xlabel('human homology class'); ax.set_ylabel('mouse homology class')
for k in range(K):                                    # mark the homologous diagonal
    ax.add_patch(plt.Rectangle((k - .5, k - .5), 1, 1, fill=False, ec='#4dd0e1', lw=1.1))
cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
cb.set_label('routed mass (row-normalised, log)')
ax.set_title('π aggregated to the 21 homology classes\n'
             f'mean self-mass {self_mass:.2f} versus {uniform:.3f} under a uniform mapping '
             f'({self_mass / uniform:.1f}×)',
             fontweight='bold', loc='left', fontsize=10.5)
plt.show()

# strongest and weakest self-correspondence — the foreshadowing of §5
d = np.diag(Crow)
order = np.argsort(d)[::-1]
print('strongest self-correspondence:', ', '.join(f'{NAMES[i]} {d[i]:.2f}' for i in order[:4]))
print('weakest  self-correspondence:', ', '.join(f'{NAMES[i]} {d[i]:.2f}' for i in order[-4:]))
print()
print('The weak end is association and limbic cortex. That is not noise — it is the coverage limit')
print('Fig. 5 quantifies and Fig. 6 turns into a disease prediction.')

## 3. Routing preserves topography (ED1d)

The clean diagonal in Fig. 1d could be a solver artefact. So test it: if two mouse parcels are close
together, are their routed human centroids close together?

For each mouse parcel we compute the transport-weighted centroid of its human mass, then correlate
pairwise mouse distances with pairwise routed-human distances. The null shuffles π's rows — which
destroys the correspondence while preserving every marginal.

In [ ]:
mouse_xyz = M.var[['x', 'y', 'z']].to_numpy(float)
human_xyz = H.var[['x', 'y', 'z']].to_numpy(float)

def routed_centroids(pi):
    w = pi / np.maximum(pi.sum(axis=1, keepdims=True), 1e-12)
    return w @ human_xyz

rng = np.random.default_rng(0)
sub = rng.choice(pi.shape[0], size=600, replace=False)          # 600 parcels -> ~180k pairs

def pdist_flat(X):
    D = np.linalg.norm(X[:, None, :] - X[None, :, :], axis=-1)
    iu = np.triu_indices(len(X), k=1)
    return D[iu]

dm = pdist_flat(mouse_xyz[sub])
dh = pdist_flat(routed_centroids(pi)[sub])
r_obs = float(pearsonr(dm, dh)[0])

null = []
for _ in range(20):
    pip = pi[rng.permutation(pi.shape[0])]
    null.append(float(pearsonr(dm, pdist_flat(routed_centroids(pip)[sub]))[0]))
null = np.array(null)

print(f'mouse pairwise distance vs routed human pairwise distance:  r = {r_obs:.2f}')
print(f'permuted-coupling null: r = {null.mean():+.3f} +/- {null.std():.3f}')
print()
print(f"canonical value (full 1,864 parcels, coupling_summary.json): "
      f"r = {cs['spatial_fidelity']['pearson_r']:.2f}, null "
      f"{cs['spatial_fidelity']['permuted_null_mean']:+.3f} +/- "
      f"{cs['spatial_fidelity']['permuted_null_sd']:.3f}")
print()
print('The subsample here reproduces the canonical r to within sampling error. The diagonal reflects')
print('preserved topography, not a solver artefact.')

In [ ]:
# ---------------- ED1d — topographic preservation ----------------
fig, ax = plt.subplots(figsize=(5.4, 4.8))
hb = ax.hexbin(dm, dh, gridsize=45, cmap='viridis', mincnt=1, bins='log')
b = np.polyfit(dm, dh, 1)
xs = np.array([dm.min(), dm.max()])
ax.plot(xs, np.polyval(b, xs), color='#c1272d', lw=2.2)
ax.set_xlabel('distance between two mouse parcels (mm)')
ax.set_ylabel('distance between their routed human centroids (mm)')
cb = fig.colorbar(hb, ax=ax, fraction=0.046, pad=0.03); cb.set_label('pairs per bin', fontsize=9)
ax.set_title(f'Routing preserves spatial topology\n'
             f"r = {cs['spatial_fidelity']['pearson_r']:.2f} against a permuted-coupling null of ≈ 0",
             fontweight='bold', loc='left', fontsize=10.5)
for s in ('top', 'right'):
    ax.spines[s].set_visible(False)
plt.show()

## 4. The evidence tiers (ED1c) — and what they actually grade

Each mouse parcel is graded on **two external lines of evidence**, neither of which is the solver's own
confidence:

- **anchored** — the parcel belongs to a curated anchor (a Garin homology class or a region pack);
- **validated** — the parcel independently reproduces a published homology (its Beauchamp region is
  enriched for routed mouse mass against a parcel-set permutation null, FDR q < 0.05).

The key finding is about *what the grade means*. Region-level recovery is essentially **equal** across
the two validated tiers (AUROC 0.87 vs 0.88). Parcel-exact recovery is **not** (top-1 0.69 vs 0.18).

**Curation buys parcel precision, not region-level correspondence.** The tier tells you the *resolution*
at which to trust a query — not whether a homologue exists at all. That is the finding §2 then explains.

In [ ]:
tiers = ev['tiers']
n = ev['n']
pct = {k: 100.0 * v / n for k, v in tiers.items()}
print(f'evidence tiers across the {n} mouse parcels')
for k in ('anchored_and_validated', 'validated_only', 'anchored_only', 'structural', 'low_evidence'):
    print(f'  {k:24s} {tiers[k]:>4d} parcels   {pct[k]:5.1f} %')
print(f'  {"":24s} {sum(tiers.values()):>4d}            {sum(pct.values()):5.1f} %   <- sums to 100')
print()
for k in pct:
    check(f'{k} %', pct[k], cs['evidence_tiers_percent'][k], 0.06)

print()
print('per-tier recovery (from coupling_summary.json):')
rec = cs['per_tier_recovery']
for k in ('anchored_and_validated', 'validated_only', 'anchored_only', 'structural', 'low_evidence'):
    d_ = rec[k]
    fields = ', '.join(f'{a} {b:.2f}' if isinstance(b, float) else f'{a} {b}'
                       for a, b in d_.items())
    print(f'  {k:24s} {fields}')
print()
print(f"the two validated tiers cover {rec['_validated_tiers_percent_of_brain']:.1f} % of the brain")

In [ ]:
# ---------------- ED1c — tier distribution ----------------
labels = ['anchored\n+ validated', 'validated\nonly', 'anchored\nonly', 'structural', 'low\nevidence']
keys = ['anchored_and_validated', 'validated_only', 'anchored_only', 'structural', 'low_evidence']
cols = ['#1b4f8a', '#4a90c2', '#e08a2b', '#b8b8b8', '#d9d9d9']
vals = [pct[k] for k in keys]

fig, ax = plt.subplots(figsize=(6.4, 3.8))
bars = ax.bar(labels, vals, color=cols, edgecolor='white')
for b_, v in zip(bars, vals):
    ax.text(b_.get_x() + b_.get_width() / 2, v + 0.6, f'{v:.0f} %', ha='center', fontsize=9)
ax.set_ylabel('mouse parcels (%)')
ax.set_ylim(0, max(vals) * 1.22)
ax.set_title(f'Evidence grade of the {n:,} mouse parcels\n'
             f"the two validated tiers cover {rec['_validated_tiers_percent_of_brain']:.0f} % of the brain",
             fontweight='bold', loc='left', fontsize=10.5)
for s in ('top', 'right'):
    ax.spines[s].set_visible(False)
plt.show()

## 5. Query the coupling (Fig. 1g)

The whole point of a probabilistic coupling: sum π over a mouse region and rank the human partners by
routed mass. Two worked examples, one cortical and one subcortical.

In [ ]:
# Parcel-level labels exist only for the curated anchors; everywhere else the Allen/Glasser label
# is "-". The meaningful query is therefore at the level of the 21 homology classes, which is what
# Fig. 1g shows: sum pi over a mouse class and rank the human classes by routed mass.

def query(mouse_class, top=4):
    i = [k for k, v in GARIN_NAMES.items() if v == mouse_class][0]
    row = Crow[i - 1]
    order = np.argsort(row)[::-1][:top]
    print(f'\nmouse {mouse_class!r} routes to:')
    for j in order:
        star = '  <- homologue' if j == i - 1 else ''
        print(f'   {100 * row[j]:5.1f} %   {NAMES[j]}{star}')

query('Motor/premotor')       # cortical seed
query('Striatum')             # subcortical seed
print()
print('Both seeds rank their expected human homologue FIRST out of 21 classes, against a uniform')
print(f'expectation of {100 / K:.1f} %. Note the honest reading: the homologue takes the top slot and a')
print('large share of the mass, but not a majority in every case -- the routed distribution has real')
print('spread, and Fig. 1g renders that spread on the human brain rather than hiding it behind an')
print('argmax. That spread is the object of study in Figs. 5 and 6.')

## 6. Summary

| property | value | source |
|---|---|---|
| coupling shape | 1,864 × 2,094 | computed |
| median top-target probability | 1.00 | `coupling_summary.json` |
| parcels with top probability > 0.5 | 92 % | `coupling_summary.json` |
| mean self-mass on the homology diagonal | 0.26 (5.5× uniform) | `fig1_coupling_matrix.json` |
| topographic fidelity | r = 0.61 vs null ≈ 0 | `coupling_summary.json` |
| evidence tiers | 31 / 22 / 13 / 14 / 21 % | `evidence_tiers_v2.json` |
| validated tiers, share of brain | 52 % | `coupling_summary.json` |

**What the tiers grade.** Region-level AUROC is equal across the two validated tiers (0.87 vs 0.88);
parcel-exact top-1 is not (0.69 vs 0.18). Curation adds *parcel* precision, not region-level
correspondence. Figure 2 shows why: connectivity and space carry *which region*; curation carries
*which parcel*.

### Panels not produced here

Figures 1a–c and ED1a,b are volumetric glass-brain renderings that need the Allen and MNI reference
volumes. They are built by `manuscript/figures/make_supervision_maps.py` and
`manuscript/figures/fig1/make_fig1_motivation.py`, from `trust_multisource_all_packs_v2.npz`.